In [22]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["KERAS_BACKEND"] = "torch"
import sys
import subprocess
from pathlib import Path
import json
import pandas as pd


# Local files/code

import src.data_preprocessing.text_preprocessing as text_pre
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
import src.config as config
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
from src.data_preprocessing.USDA_processing import embed_USDA_data
import src.evaluation.visual_evaluator as visual_evaluator
import src.data_preprocessing.image_preprocessing as img_pre
from src.agents.NLP_Agent import AnswerRanker, build_answer_index, add_usda_records
import src.evaluation.text_evaluator as text_eval

In [23]:
#load in the GBIF data
#GET THE PATH TO ROOT OF PROJECT
ROOT_PATH = Path.cwd()
print(ROOT_PATH)
TRAIN_DATA = pd.read_csv(config.Data.GBIF.TRAIN_FILE)
TEST_DATA = pd.read_csv(config.Data.GBIF.TEST_FILE)
VAL_DATA = pd.read_csv(config.Data.GBIF.VALIDATION_FILE)

c:\Users\Kyler_Code\Desktop\PlantQA


In [24]:
##Iterate Through Data sets and fix pathing issue

TRAIN_DATA["image_path"] = TRAIN_DATA["image_path"].apply(lambda x: x.lstrip('//'))
TRAIN_DATA["image_path"] = TRAIN_DATA["image_path"].apply(lambda x: x.replace(r"/", "\\"))
#TRAIN_DATA["image_path"] = TRAIN_DATA["image_path"].apply(lambda p: str(ROOT_PATH / p))


TEST_DATA["image_path"] = TEST_DATA["image_path"].apply(lambda x: x.lstrip('//'))
TEST_DATA["image_path"] = TEST_DATA["image_path"].apply(lambda x: x.replace(r"/", "\\"))
#TEST_DATA["image_path"] = TEST_DATA["image_path"].apply(lambda p: str(ROOT_PATH / p))

VAL_DATA["image_path"] = VAL_DATA["image_path"].apply(lambda x: x.lstrip(r'/'))
VAL_DATA["image_path"] = VAL_DATA["image_path"].apply(lambda x: x.replace(r"/", "\\"))
#VAL_DATA["image_path"] = VAL_DATA["image_path"].apply(lambda p: str(ROOT_PATH / p))




In [25]:
for i, r in TRAIN_DATA.iterrows():
    print(r["image_path"])

data\GBIF_Pre_Processed\CALA2\CALA2_105.jpeg
data\GBIF_Pre_Processed\ASSE12\ASSE12_48.jpg
data\GBIF_Pre_Processed\CAPS5\CAPS5_55.jpg
data\GBIF_Pre_Processed\CAPE6\CAPE6_74.jpg
data\GBIF_Pre_Processed\CACR7\CACR7_0.jpg
data\GBIF_Pre_Processed\VILO6\VILO6_6.jpg
data\GBIF_Pre_Processed\BEHI3\BEHI3_13.jpeg
data\GBIF_Pre_Processed\CAAN11\CAAN11_94.jpg
data\GBIF_Pre_Processed\ASVI2\ASVI2_43.jpg
data\GBIF_Pre_Processed\CACA18\CACA18_7.jpg
data\GBIF_Pre_Processed\CALE6\CALE6_15.jpg
data\GBIF_Pre_Processed\TRUN\TRUN_37.jpg
data\GBIF_Pre_Processed\AQEL\AQEL_51.jpg
data\GBIF_Pre_Processed\BRPO2\BRPO2_14.jpg
data\GBIF_Pre_Processed\VEAG\VEAG_78.jpeg
data\GBIF_Pre_Processed\WYMO\WYMO_30.jpg
data\GBIF_Pre_Processed\BOHI\BOHI_85.jpg
data\GBIF_Pre_Processed\AMGL2\AMGL2_5.jpg
data\GBIF_Pre_Processed\ACCH5\ACCH5_26.jpeg
data\GBIF_Pre_Processed\CACR7\CACR7_118.jpeg
data\GBIF_Pre_Processed\ASHA13\ASHA13_25.jpg
data\GBIF_Pre_Processed\VIRH\VIRH_48.jpg
data\GBIF_Pre_Processed\ARRU4\ARRU4_4.jpg
data\GBIF_Pre

In [26]:
traits = ["crop", "disease", "severity"]


train_img = VisualModel.one_row_per_image(TRAIN_DATA, traits)
val_img = VisualModel.one_row_per_image(TEST_DATA, traits)
test_img = VisualModel.one_row_per_image(VAL_DATA, traits)
classes = VisualModel.build_classes(train_img, traits)


train_ds = VisualModel.make_dataset(train_img, classes, config.General.PROJECT_ROOT, training=True, use_mask=False)
val_ds = VisualModel.make_dataset(val_img, classes, config.General.PROJECT_ROOT, use_mask=False)

In [27]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch
print(keras.backend.backend())
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

torch
True
NVIDIA GeForce RTX 5070


In [28]:
keras.mixed_precision.set_global_policy("mixed_bfloat16")

In [29]:

visual_model = VisualModel(classes=classes, use_mask=False)
visual_model.build()
visual_model.compile()
history_frozen = visual_model.fit(train_ds, val_ds, epochs=30)
visual_model.unfreeze_layers()
history_tuned = visual_model.fit(train_ds, val_ds, epochs=5)

visual_evaluator.plot_loss(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned, heads=["crop", "disease"])

import json
Path("./models/visual_models/9_22_classes.json").write_text(json.dumps(classes))
visual_model.save(path="./models/visual_models/9_24.keras")
 

Epoch 1/30
3414/3414 ━━━━━━━━━━━━━━━━━━━━ 557s 163ms/step - crop_accuracy: 0.1400 - crop_loss: 5.4080 - disease_accuracy: 0.9996 - disease_loss: 0.0025 - loss: 5.4134 - severity_accuracy: 0.9993 - severity_loss: 0.0030 - val_crop_accuracy: 0.2434 - val_crop_loss: 4.3515 - val_disease_accuracy: 1.0000 - val_disease_loss: 9.1265e-05 - val_loss: 4.3517 - val_severity_accuracy: 1.0000 - val_severity_loss: 9.0842e-05
Epoch 2/30
3414/3414 ━━━━━━━━━━━━━━━━━━━━ 526s 154ms/step - crop_accuracy: 0.2949 - crop_loss: 3.8481 - disease_accuracy: 1.0000 - disease_loss: 4.7402e-05 - loss: 3.8482 - severity_accuracy: 1.0000 - severity_loss: 5.3781e-05 - val_crop_accuracy: 0.2907 - val_crop_loss: 3.9592 - val_disease_accuracy: 1.0000 - val_disease_loss: 1.9925e-05 - val_loss: 3.9592 - val_severity_accuracy: 1.0000 - val_severity_loss: 2.0501e-05
Epoch 3/30
3414/3414 ━━━━━━━━━━━━━━━━━━━━ 530s 155ms/step - crop_accuracy: 0.3645 - crop_loss: 3.3051 - disease_accuracy: 1.0000 - disease_loss: 9.8877e-06 - lo

KeyboardInterrupt: 